### Assignment 1

1. Data cleaning

Most common cleanup procedures are: removal of incorrect information, handling missing values, noise
removal, handling inconsistencies, writing to numeric, and detecting outliers. After clean up, data is
integrated, transformed and reduced. The aim of the procedures is to create attributes that are easier
to analyze with software, remove redundant information, and fit information in smaller space.
- (a) Consider the bus data collection. How many lines of data are collected during one day if each
buslocation is reported once a second? Give a rough estimate.
- (b) Load file rawData.arff to Weka and use ‘Explorer’ to analyze the file. In theory, samples are taken
twice a second, but what is the real sampling rate? There are 50000 lines of data in the file, how long did
it take to collect the data? How many vehicles are included in the file (each ‘VehicleRef’ indicates one
bus)?
- (c) What are maximum, minimum and stDev of delay in seconds? Which bus line has the
maximumvalue?
- (d) What is an outlier? How does one define outliers on delay values? Where would you put a
thresholdand why?
- (e) Explain what could be done to attributes ‘OperatorRef’, ‘DestinationName’, and ‘OriginName’
tomake data more useable? (Use ‘Edit’ and ‘Filter’ to make these changes to attributes)
- (d) In this exercise the end goal is to consider delay analysis. For automatic analysis tasks and
classification Weka needs all of the attributes to be of types: numeric, nominal, or date. Remove unusable
attributes. How many attributes are left?
2. Attribute selection

- (a) Use 'Select attributes' to evaluate the significance of attributes. Choose 'CfSubsetEval' as
theattribute evaluator and 'BestFirst' with default settings as the search method. Select ‘Delay_Seconds’
as the attribute to be analyzed. How does the evaluator select the subset? How does the method
evaluate significance?
- (b) Run the attribute selection and report the selection. What causes them to be selected? Does
theselection make sense?
- (c) Consider the sample size, does it show in the selected attributes? Consider the previous answers
andthe selection of attributes by Weka. Which attributes would you use for delay analysis?

#### Part 1:  Data cleaning

a) First we find the number of lines per day for one bus, and multiply it by the total number of buses.
Receiving **86400** for a single bus and around **11318400** lines in total created daily for all the buses in tampere.

However, since the number of buses and actual working hours may differ, this is likely far from the actual number


In [1]:
single_buss_count = 24 * 60 * 60
total_count = single_buss_count * 131 #the number is found later
print(single_buss_count)
print(total_count)

86400
11318400


b) Actual sample rate appears to be around **1.26** times per second. It took **303** seconds to collect all the data. There are **131** vehicles included in the file

In [2]:
import pandas as pd

df = pd.read_csv("rawData.arff", skiprows=28, header=None, encoding="latin1")

columns = [
    'ResponseTimeDate', 'ResponseTimestamp', 'ProducerRef', 'ValidUntilTimeDate', 'ValidUntilTimeStamp',
    'RecordedAtTimeDate', 'RecordedAtTimeStamp', 'LineRef', 'DirectionRef', 'DataFrameRef',
    'DatedVehicleJourneyRef', 'OperatorRef', 'OriginName', 'OriginNameLanguage',
    'DestinationName', 'DestinationNameLanguage', 'Monitored', 'VehicleLocationLongitude',
    'VehicleLocationLatitude', 'Bearing', 'Delay_String', 'Delay_Seconds', 'VehicleRef'
]
df.columns = columns

df['ResponseTimeDate'] = pd.to_datetime(df['ResponseTimeDate'].str.strip('"'))
start = df['ResponseTimeDate'].min()
end = df['ResponseTimeDate'].max()
duration = (end - start).total_seconds()

vehicle_count = df['VehicleRef'].nunique()
sampling_rate = (len(df) / vehicle_count) / duration

print(round(sampling_rate, 2))
print(round(duration, 2))
print(vehicle_count)


1.26
303.0
131


c) The delay values are as follows: max: **3594**, min: **-268**, stDev: **404**; The bus line #**13** had the biggest delay.

In [3]:
max_delay = df['Delay_Seconds'].max()
min_delay = df['Delay_Seconds'].min()
std_delay = df['Delay_Seconds'].std()

line_with_max_delay = df.loc[df['Delay_Seconds'].idxmax(), 'LineRef']

print(max_delay)
print(min_delay)
print(round(std_delay, 2))
print(line_with_max_delay)

3594
-268
404.79
 "13"


d) Outliers are the data points that significantly differ from the rest of the data. For example, in the case of the delays, outliers are the buses that arrive significantly earlier or later. The threshold is usually put as **1.5*IRQ** meaning everything less than **25%** and more than **75%** is an outlier.

e) To make OperatorRef, OriginName, and DestinationName more usable, text format should be standardized, encoding should be fixed, and missing values changed to “unknown”.

In [4]:
import unicodedata

df['OperatorRef'] = df['OperatorRef'].str.strip().str.lower().fillna('unknown')

df['OriginName'] = df['OriginName'].str.strip().fillna('unknown').apply(
    lambda x: unicodedata.normalize('NFKD', x).encode('ascii', 'ignore').decode('ascii')
)
df['DestinationName'] = df['DestinationName'].str.strip().fillna('unknown').apply(
    lambda x: unicodedata.normalize('NFKD', x).encode('ascii', 'ignore').decode('ascii')
)

f) To prepare data for weka, we drop all attributes that are not numeric, nominal or useful for delay analysis.
specifically, we removed:
- timestamp strings: **ResponseTimeDate**, **ValidUntilTimeDate**, **RecordedAtTimeDate**
- metadata: **Service producer reference**, **ProducerRef**, **VehicleRef**
- language & boolean flags: **OriginNameLanguage**, **DestinationNameLanguage**, **Monitored**
- overly unique fields: **DataFrameRef**, **DatedVehicleJourneyRef**
- and the text-based delay column: **Delay_String**
After the cleanup, **12** attributes left.

In [5]:
drop_cols = [
    'ResponseTimeDate', 'ValidUntilTimeDate', 'RecordedAtTimeDate',
    'DataFrameRef', 'DatedVehicleJourneyRef',
    'OriginNameLanguage', 'DestinationNameLanguage',
    'Monitored', 'Delay_String', 'VehicleRef',
    'ProducerRef'
]


df_cleaned = df.drop(columns=drop_cols)

 #aslo removing embedded quotes
for col in df_cleaned.select_dtypes(include='object').columns:
    df_cleaned[col] = df_cleaned[col].str.replace('"', '', regex=False).str.strip()

df_cleaned.shape

(49998, 12)

#### Part 2: Attribute selection

a) The evaluator selects a subset of attributes that together explain the delay in the best way, looks how good the group correlates with the target, and at the same tome minimizing their correlation.

The method adds features one by one and keeps only those that are score best.

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression

X = df_cleaned.drop(columns=['Delay_Seconds'])
y = df_cleaned['Delay_Seconds'].astype(float)

categorical_cols = ['LineRef', 'DirectionRef', 'OperatorRef', 'OriginName', 'DestinationName']

preprocessor = ColumnTransformer(
    [('oh', OneHotEncoder(sparse_output=False, drop='first'), categorical_cols)],
    remainder='passthrough'
)

sfs = SequentialFeatureSelector(
    LinearRegression(),
    n_features_to_select=10,
    direction='forward',
    scoring='r2',
    cv=5
)

pipe = make_pipeline(preprocessor, sfs)
pipe.fit(X, y)

mask = pipe.named_steps['sequentialfeatureselector'].get_support()
ft_names = pipe.named_steps['columntransformer'].get_feature_names_out(input_features=X.columns)
selected = ft_names[mask]

print("Selected features:", list(selected))

Selected features: ['oh__LineRef_11', 'oh__LineRef_18', 'oh__LineRef_31', 'oh__LineRef_37', 'oh__OriginName_Haukiluoma', 'oh__OriginName_Keskustori R', 'oh__OriginName_Teralahti', 'oh__DestinationName_Haukiluoma', 'oh__DestinationName_IKEA', 'remainder__VehicleLocationLongitude']


b) The selected features are:  **['LineRef_11', 'LineRef_18', 'LineRef_31', 'LineRef_37',
'OriginName_Haukiluoma', 'OriginName_Keskustori R', 'OriginName_Teralahti',
'DestinationName_Haukiluoma', 'DestinationName_IKEA', 'VehicleLocationLongitude']**

They were likely chosen because certain bus lines and stops have more consistent delays,  and the location probably shows traffic-heavy areas with common delays.

The selection makes sense, and seems valid to predict delays.

c) I think the sample size is big enough to show the patterns. For delay analysis, I would probably use the following attributes: **LineRef, DirectionRef, OperatorRef, OriginName, DestinationName, VehicleLocationLongitude, VehicleLocationLatitude**.